# E9d report: control vs balanced (A5000)

Reads `<STAGED_PINN_RESULTS>/<E9D_TAG>/*/summary.json`. Re-executed by the queue after each arm; it can be run at any time, also while an arm is still running.

In [ ]:
import json, os, math
from pathlib import Path
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
pd.set_option("display.precision", 4); pd.set_option("display.width", 160)
TAG = os.environ.get("E9D_TAG", "e9d")
ROOT = Path(os.environ.get("STAGED_PINN_RESULTS", "Results")) / TAG
arms = {p.parent.name: json.loads(p.read_text()) for p in sorted(ROOT.glob("*/summary.json"))}
print(ROOT, "->", {k: v.get("status") for k, v in arms.items()})

## Summary per arm (validation, 175 devices; curve/Jsc in mA/cm2, PCE/FF in pp, Voc in V)

In [ ]:
METRICS = ("curve", "curve_to_voc", "Jsc", "Voc", "Voc_fine", "Voc_bias_mV", "FF", "PCE", "invalid_Voc", "invalid_Voc_fine")
rows = []
for name, a in arms.items():
    for phase in ("pre_validation", "adam_validation", "final"):
        if phase in a:
            v = a[phase]["jv"]["all"]
            rows.append({"arm": name, "phase": phase.replace("_validation", ""), **{m: v.get(m) for m in METRICS},
                         **{k: a[phase]["clean_dense_terms"].get(k) for k in ("P", "Kn", "Kp", "Cn", "Cp")}})
    if "best" in a:
        rows.append({"arm": name, "phase": f"best@{a['best']['phase']}{a['best']['step']}", "curve": a["best"]["curve"]})
summary = pd.DataFrame(rows)
summary

## Cost

In [ ]:
cost = []
for name, a in arms.items():
    ck = [c for c in a.get("checkpoints", []) if c["phase"] == "adam"]
    per = ck[-1]["seconds"] / ck[-1]["step"] if ck else math.nan
    cost.append({"arm": name, "status": a.get("status"), "devices": a.get("devices"), "lb_devices": a.get("lb_devices"),
                 "physics": a.get("physics", True), "adam_s_per_step": per,
                 "adam_h_projected_4000": per * 4000 / 3600 if ck else math.nan,
                 "lbfgs_evals": a.get("lbfgs_evaluations"), "lbfgs_s": a.get("lbfgs_seconds"),
                 "lbfgs_s_per_eval": (a.get("lbfgs_seconds") or math.nan) / max(a.get("lbfgs_evaluations") or 1, 1),
                 "peak_GB": a.get("peak_memory_gb"), "init": a.get("initial_weights", "")[:10]})
pd.DataFrame(cost)

## Validation curve error during training

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, a in arms.items():
    ck = a.get("checkpoints", [])
    pre = max([c["step"] for c in ck if c["phase"] == "pre"], default=0)
    x = [c["step"] + (pre if c["phase"] == "adam" else 0) for c in ck]
    axes[0].plot(x, [c["validation"]["all"]["curve"] for c in ck], label=name)
    axes[1].plot(x, [c["validation"]["all"]["PCE"] for c in ck], label=name)
for ax, t in zip(axes, ("curve RMSE (mA/cm2)", "PCE error (pp)")):
    ax.set_yscale("log"); ax.set_xlabel("update (PRE then Adam)"); ax.set_title(t); ax.legend(); ax.grid(alpha=.3)
fig.tight_layout(); fig.savefig(ROOT / "curves.png", dpi=120); plt.show()

## Final metrics per material

In [ ]:
rows = []
for name, a in arms.items():
    if "final" in a:
        for mat, v in a["final"]["jv"].items():
            rows.append({"arm": name, "material": mat, **{m: v.get(m) for m in METRICS}, "n": v.get("n")})
pd.DataFrame(rows)

## Fair comparison check

`control_noise0_seed0` and `balanced_noise0_seed0` must share initial weights; the only difference is the physics terms.

In [ ]:
c, b = arms.get("control_noise0_seed0"), arms.get("balanced_noise0_seed0")
if c and b:
    print("same init:", c["initial_weights"] == b["initial_weights"])
    lc, lb = c["pre_calibration"]["lambda"], b["pre_calibration"]["lambda"]
    print({k: (lc[k], lb[k]) for k in lc if lc[k] != lb[k]})
pd.DataFrame(rows).to_csv(ROOT / "final_per_material.csv", index=False); summary.to_csv(ROOT / "summary.csv", index=False)